In [ ]:
exec(open(__import__('pathlib').Path(__vsc_ipynb_file__).parent.parent / 'src' / 'display_html.py').read())

## T_PairEnv / T_PairDataset — Pair Structure Task

**Role**: Schapiro (2017) §3.a control task — sequential random walk over 4 pairs (AB/CD/EF/GH).  
Both A and B items appear as ECin. A items transition deterministically to their B partner;  
B items transition uniformly to the A item of one of the other 3 pairs.  
Back-to-back repetition prevented by excluding own pair from B's choices.  
80 inputs/epoch (Schapiro 2017 §3.a).

**Graph structure**:
- A items (0, 2, 4, 6): A → B partner with probability 1.0
- B items (1, 3, 5, 7): B → A of any other pair with probability 1/3 each (3 choices)

**Why pairs?**  
In contrast to the community graph, the pair task has no higher-order statistical structure —  
only pairwise associations. The B→A transitions are fully random, so MSP cannot exploit  
community-level regularities. Used to dissociate MSP from TSP contributions.

**Schapiro (2017) p.4**: "After AB, BC, BE, or BG followed with equal probability;  
if BC was chosen, the next input would be CD."

In [ ]:
# path & directories
import sys
from pathlib import Path

SRC = str((Path(__vsc_ipynb_file__).parent.parent / 'src').resolve())
VIZ = (Path(__vsc_ipynb_file__).parent.parent / 'visualizations').resolve()
sys.path.insert(0, SRC)

# hyperparameters
import torch
import numpy as np
import matplotlib.pyplot as plt
from tasks import T_PairEnv, T_PairDataset

# Schapiro (2017) §3.a task parameters
N_PAIRS = 4    # AB / CD / EF / GH
N_STEPS = 800  # 80 inputs/epoch × 10 epochs (Schapiro 2017 §3.a)
N_ITEMS = N_PAIRS * 2
SEED    = 0

In [ ]:
# Instantiate and inspect structure
env = T_PairEnv(n_pairs=N_PAIRS, seed=SEED)

print(f"n_pairs : {env.n_pairs}")
print(f"n_items : {env.n_items}")
print(f"pairs   : {env.pairs}")
print()
print("Transition structure (Schapiro 2017 p.4):")
for i, (a, b) in enumerate(env.pairs):
    other_a = [env.pairs[j][0] for j in range(N_PAIRS) if j != i]
    print(f"  Pair {i}: {a}→{b} (P=1.0),   {b}→{other_a} (P=1/3 each)")

In [ ]:
# Run 20 steps: verify A→B (deterministic) and B→other_A (3 choices, own pair excluded)
env.reset(seed=0)
transitions = [env.step() for _ in range(20)]

print("step  cur  nxt  type                valid?")
all_ok = True
for i, (cur, nxt) in enumerate(transitions):
    pair_idx = cur // 2
    if cur % 2 == 0:           # A item
        valid  = (nxt == cur + 1)
        t_type = f"A→B  (pair {pair_idx})"
    else:                      # B item
        valid  = (nxt % 2 == 0) and (nxt // 2 != pair_idx)
        t_type = f"B→A  (pair {pair_idx}→{nxt // 2})"
    if not valid:
        all_ok = False
    print(f"  {i:2d}    {cur:2d}   {nxt:2d}  {t_type:<22}  {valid}")

print(f"\nAll transitions valid: {all_ok}  (expect True)")

# No back-to-back: B never returns to own pair's A
back_to_back = any(
    transitions[i][1] // 2 == transitions[i][0] // 2
    for i in range(len(transitions))
    if transitions[i][0] % 2 == 1   # B items only
)
print(f"Back-to-back same pair via B→A: {back_to_back}  (expect False)")

In [ ]:
# Transition statistics over N_STEPS
# A items: P(A→B) = 1.0; B items: P(B→each other A) ≈ 1/3
env.reset(seed=42)
counts = np.zeros((N_ITEMS, N_ITEMS), dtype=int)
for _ in range(N_STEPS):
    cur, nxt = env.step()
    counts[cur, nxt] += 1

print("A items — P(A→B) should be 1.0:")
for a in range(0, N_ITEMS, 2):
    b     = a + 1
    total = counts[a].sum()
    print(f"  Item {a} → item {b}: {counts[a, b]}/{total} ({100*counts[a,b]/max(total,1):.0f}%)")

print()
print("B items — P(B→each other A) should be ≈ 1/3 (own pair's A = 0):")
for b in range(1, N_ITEMS, 2):
    pair_idx = b // 2
    total    = counts[b].sum()
    parts    = []
    for j in range(N_PAIRS):
        a    = j * 2
        pct  = 100 * counts[b, a] / max(total, 1)
        mark = " (own)" if j == pair_idx else ""
        parts.append(f"→{a}: {pct:.0f}%{mark}")
    print(f"  Item {b} (pair {pair_idx}):  " + "  ".join(parts))

In [ ]:
# T_PairDataset: verify shapes and transition correctness
dataset = T_PairDataset(n_steps=N_STEPS, n_pairs=N_PAIRS, seed=7)

print(f"Dataset length         : {len(dataset)}")
sample = dataset[0]
print(f"Keys                   : {list(sample.keys())}")
print(f"item_onehot shape      : {sample['item_onehot'].shape}")
print(f"target_onehot shape    : {sample['target_onehot'].shape}")
print()

# Count A→B and B→A_other transitions; verify all are valid
n_a_to_b, n_b_to_a = 0, 0
all_valid = True
for i in range(len(dataset)):
    s        = dataset[i]
    cur      = s['item'].item()
    nxt      = s['next_item'].item()
    pair_idx = cur // 2
    if cur % 2 == 0:
        n_a_to_b += 1
        if nxt != cur + 1:
            all_valid = False
    else:
        n_b_to_a += 1
        if nxt % 2 != 0 or nxt // 2 == pair_idx:
            all_valid = False

print(f"A→B steps     : {n_a_to_b}  (expect ~{N_STEPS // 2})")
print(f"B→A_other steps: {n_b_to_a}  (expect ~{N_STEPS // 2})")
print(f"All transitions valid: {all_valid}  (expect True)")

# A→B counts per pair (should be roughly equal)
pair_counts = torch.zeros(N_PAIRS, dtype=torch.long)
for i in range(len(dataset)):
    s = dataset[i]
    if s['item'].item() % 2 == 0:
        pair_counts[s['pair']] += 1
print(f"\nA→B counts per pair (expect ~{N_STEPS // N_PAIRS // 2} each): {pair_counts.tolist()}")

## Without statistical learning: interleaved condition

Schapiro (2017) p.4: "AB, CD, EF, and GH all appeared but **never BC or FG**."  
Pairs are presented as isolated units; no transitions connect them.  
Only A items appear as ECin — the network memorizes fixed A→B associations directly.  
No statistics need to be accumulated: TSP can bind each pair episodically.

In [ ]:
# Interleaved mode: B items never appear as current; no between-pair transitions
env_il = T_PairEnv(n_pairs=N_PAIRS, interleaved=True, seed=SEED)
env_il.reset()

transitions_il = [env_il.step() for _ in range(20)]
print("Interleaved — first 20 steps:")
print("step  cur  nxt  pair  only_A_as_current?")
all_a_only = True
for i, (cur, nxt) in enumerate(transitions_il):
    is_a = (cur % 2 == 0)
    if not is_a:
        all_a_only = False
    print(f"  {i:2d}    {cur:2d}   {nxt:2d}    {cur // 2}    {is_a}")
print(f"\nOnly A items as current: {all_a_only}  (expect True)")

# Transition counts: A→B only; B→anything should be 0
env_il.reset(seed=42)
counts_il = np.zeros((N_ITEMS, N_ITEMS), dtype=int)
for _ in range(N_STEPS):
    cur, nxt = env_il.step()
    counts_il[cur, nxt] += 1

b_transitions = counts_il[1::2].sum()
print(f"B items used as current (expect 0): {b_transitions}")
print(f"Between-pair transitions (expect 0): {sum(counts_il[a, b] for a in range(0,N_ITEMS,2) for b in range(N_ITEMS) if b != a+1)}")

# Compare dataset outputs: interleaved vs sequential
ds_seq = T_PairDataset(n_steps=N_STEPS, n_pairs=N_PAIRS, interleaved=False, seed=0)
ds_il  = T_PairDataset(n_steps=N_STEPS, n_pairs=N_PAIRS, interleaved=True,  seed=0)

seq_b_count = sum(1 for i in range(len(ds_seq)) if ds_seq[i]['item'].item() % 2 == 1)
il_b_count  = sum(1 for i in range(len(ds_il))  if ds_il[i]['item'].item()  % 2 == 1)
print(f"\nSequential dataset — B items as current : {seq_b_count}  (expect ~{N_STEPS // 2})")
print(f"Interleaved dataset — B items as current: {il_b_count}   (expect 0)")

In [ ]:
# Transition probability heatmaps: sequential vs interleaved
T_prob    = counts    / counts.sum(axis=1,    keepdims=True).clip(1)
T_prob_il = counts_il / counts_il.sum(axis=1, keepdims=True).clip(1)
item_labels = list('ABCDEFGH')[:N_ITEMS]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, mat, title, cmap in [
    (axes[0], T_prob,    'Sequential\n(with statistical learning)',     'Blues'),
    (axes[1], T_prob_il, 'Interleaved\n(without statistical learning)', 'Oranges'),
]:
    im = ax.imshow(mat, vmin=0, vmax=1, cmap=cmap, aspect='auto')
    ax.set_xticks(range(N_ITEMS))
    ax.set_yticks(range(N_ITEMS))
    ax.set_xticklabels(item_labels)
    ax.set_yticklabels(item_labels)
    ax.set_xlabel('Next item')
    ax.set_ylabel('Current item')
    ax.set_title(title, fontsize=10)
    for i in range(N_ITEMS):
        for j in range(N_ITEMS):
            v = mat[i, j]
            if v > 0:
                ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                        fontsize=7.5, color='white' if v > 0.5 else 'black')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle('Pair task transition probabilities (Schapiro 2017 §3.a)', fontsize=11)
plt.tight_layout()
plt.savefig(VIZ / 'pair_task_transitions.png', dpi=150, bbox_inches='tight')
plt.show()